In [35]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier

# 1. Load data
data = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

test_ids = test["PassengerId"]

# 2. Clean data
def clean(df):
    df = df.drop(["Ticket", "Cabin", "Name", "PassengerId"], axis=1)

    for col in ["SibSp", "Parch", "Fare", "Age"]:
        df[col] = df[col].fillna(df[col].median())

    df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])
    return df

data = clean(data)
test = clean(test)

# Feature Engineering new

# Family size
data["FamilySize"] = data["SibSp"] + data["Parch"]
test["FamilySize"] = test["SibSp"] + test["Parch"]

# IsAlone
data["IsAlone"] = (data["FamilySize"] == 0).astype(int)
test["IsAlone"] = (test["FamilySize"] == 0).astype(int)


# 4. Split X / y
y = data["Survived"]
X = data.drop("Survived", axis=1)

# 5. Encoding
for col in ["Sex", "Embarked"]:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    test[col] = le.transform(test[col])

# 6. Train / validation split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 7. Model 
model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    max_depth=5
)

model.fit(X_train, y_train)

# 8. Predict
predictions = model.predict(test)

# 9. Save submission
submission = pd.DataFrame({
    "PassengerId": test_ids,
    "Survived": predictions
})

submission.to_csv("submission.csv", index=False)